<div align="center">

# COMSATS UNIVERSITY ISLAMABAD
### Department of Computer Science / Artificial Intelligence

## NATURAL LANGUAGE PROCESSING
### LAB 05  |  N Gram Language Models

</div>

| **Student Information** | **Details** |
|:---|:---|
| **Name** | Ayesha Mehmood |
| **Registration No.** | FA23-BAI-010 |
| **Instructor** | Sir Zaeem Akbar Khan |
| **University** | COMSATS University Islamabad |
| **Lab Number** | Lab 05 |

---

**Instructions:** Run each code cell from top to bottom. Sections follow the official lab task sheet. Example inputs are for demonstration, and can be changed as needed.

## Lab Tasks


---
## Task 1: Tokenization


In [ ]:
# Training and testing data
train = [
    "the cat sat on the mat",
    "the cat ate the fish",
    "the dog sat on the rug",
    "the dog ate the bone"
]

test = [
    "the cat sat on the rug",
    "the fish sat on the mat"
]


# 1. Write tokenize(sentence) that lowercases, splits on spaces and adds <s> and </s>.

def tokenize(sentence):
    return ["<s>"] + sentence.lower().split(" ") + ["</s>"]


# 2. Apply it to every training and test sentence and print the results.

print("Training Sentences:")
for sentence in train:
    print(tokenize(sentence))

print("\nTest Sentences:")
for sentence in test:
    print(tokenize(sentence))


# 3. Print the total number of tokens in the training corpus with markers and without markers.

with_markers = sum(len(tokenize(sentence)) for sentence in train)
without_markers = sum(len(sentence.split(" ")) for sentence in train)

print("\nTotal tokens with markers:", with_markers)
print("Total tokens without markers:", without_markers)


# 4. Write build_vocab(sentences) and print V and the sorted vocabulary.

def build_vocab(sentences):
    vocab = set()

    for sentence in sentences:
        vocab.update(tokenize(sentence))

    vocab.discard("<s>")
    return vocab

vocab = build_vocab(train)
V = len(vocab)

print("\nVocabulary Size (V):", V)
print("Sorted Vocabulary:", sorted(vocab))

---
## Task 2: Generating N grams


In [ ]:
# 1. Write get_ngrams(tokens, n) that returns a list of tuples.

def get_ngrams(tokens, n):
    if n < 1:
        return []
    return [tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1)]


# 2. Print the unigrams, bigrams and trigrams of the first training sentence.

tokens = tokenize(train[0])

print("Unigrams:")
print(get_ngrams(tokens, 1))

print("\nBigrams:")
print(get_ngrams(tokens, 2))

print("\nTrigrams:")
print(get_ngrams(tokens, 3))


# 3. For every sentence, verify with code that m tokens give
# m - n + 1 N-grams for n = 1, 2 and 3.

print("\nVerifying N-gram Counts:")

for sentence in train + test:
    tokens = tokenize(sentence)
    m = len(tokens)

    for n in [1, 2, 3]:
        actual = len(get_ngrams(tokens, n))
        expected = m - n + 1

        assert actual == expected

        print(f"Sentence: {sentence}")
        print(f"n={n}, Expected={expected}, Actual={actual}")

print("\nAll N-gram counts verified successfully!")


# 4. Show what get_ngrams returns when n is larger than
# the sentence length, and make sure it does not crash.

tokens = tokenize(train[0])

n = len(tokens) + 1

result = get_ngrams(tokens, n)

print("\nWhen n is larger than sentence length:")
print("Tokens:", tokens)
print("n:", n)
print("Result:", result)

---
## Task 3: Counting N grams


In [ ]:
# 1. Use collections.Counter to build unigram, bigram
# and trigram counts for the training corpus.

from collections import Counter

unigram_counts = Counter()
bigram_counts = Counter()
trigram_counts = Counter()

for sentence in train:
    tokens = tokenize(sentence)

    unigram_counts.update(get_ngrams(tokens, 1))
    bigram_counts.update(get_ngrams(tokens, 2))
    trigram_counts.update(get_ngrams(tokens, 3))


# 2. Print each count table sorted from highest to lowest count.

print("Unigram Counts:")
for ngram, count in unigram_counts.most_common():
    print(ngram, ":", count)

print("\nBigram Counts:")
for ngram, count in bigram_counts.most_common():
    print(ngram, ":", count)

print("\nTrigram Counts:")
for ngram, count in trigram_counts.most_common():
    print(ngram, ":", count)


# 3. Print the 5 most common bigrams and the number of
# distinct unigrams, bigrams and trigrams.

print("\n5 Most Common Bigrams:")
for ngram, count in bigram_counts.most_common(5):
    print(ngram, ":", count)

print("\nNumber of Distinct N-grams:")
print("Unigrams:", len(unigram_counts))
print("Bigrams:", len(bigram_counts))
print("Trigrams:", len(trigram_counts))


# 4. Write count_of(ngram) that returns 0 for an unseen
# N-gram instead of raising an error.
# Test it on one seen and one unseen bigram.

def count_of(ngram):
    n = len(ngram)

    if n == 1:
        return unigram_counts.get(ngram, 0)
    elif n == 2:
        return bigram_counts.get(ngram, 0)
    elif n == 3:
        return trigram_counts.get(ngram, 0)
    else:
        return 0

print("\nTesting count_of():")

print("Seen Bigram ('the', 'cat'):",
      count_of(('the', 'cat')))

print("Unseen Bigram ('fish', 'sat'):",
      count_of(('fish', 'sat')))

---
## Task 4: Unigram Model


In [ ]:
# 1. Write unigram_prob(w) using MLE, P(w) = C(w) / N.

from itertools import permutations

N = sum(unigram_counts.values())

def unigram_prob(w):
    return unigram_counts.get((w,), 0) / N


# 2. Print all unigram probabilities as a table and
# assert that they sum to 1.

print("Unigram Probability Table:")
print(f"{'Word':<12} {'Count':<10} {'Probability':<12}")

for word in sorted(vocab):
    count = unigram_counts.get((word,), 0)
    prob = unigram_prob(word)
    print(f"{word:<12} {count:<10} {prob:.4f}")

total_prob = sum(unigram_prob(w[0]) for w in unigram_counts)

assert abs(total_prob - 1.0) < 1e-9

print("\nSum of probabilities:", round(total_prob, 4))


# 3. Write unigram_sentence_prob(sentence) and
# print the result for both test sentences.

def unigram_sentence_prob(sentence):
    tokens = tokenize(sentence)
    tokens = tokens[1:]  # Exclude <s>

    probability = 1.0

    for word in tokens:
        probability *= unigram_prob(word)

    return probability


print("\nUnigram Sentence Probabilities:")

for sentence in test:
    prob = unigram_sentence_prob(sentence)
    print(f"{sentence}: {prob:.4f}")


# 4. Use itertools.permutations on the words of the first
# test sentence and show that different word orders
# can give the same unigram probability.

words = test[0].split(" ")

all_permutations = permutations(words)

print("\nDifferent Word Orders with Same Unigram Probability:")

shown = set()
count = 0

for perm in all_permutations:
    sentence = " ".join(perm)

    if sentence not in shown:
        shown.add(sentence)

        prob = unigram_sentence_prob(sentence)
        print(f"{sentence}: {prob:.4f}")

        count += 1

    if count == 5:
        break

---
## Task 5: Bigram Model (MLE)


In [ ]:
# 1. Write bigram_prob(prev, w) using
# P(w | v) = C(v, w) / C(v).

def bigram_prob(prev, w):
    bigram_count = bigram_counts.get((prev, w), 0)
    prev_count = unigram_counts.get((prev,), 0)

    if prev_count == 0:
        return 0.0

    return bigram_count / prev_count


# 2. Write next_word_distribution(prev) that returns
# a dictionary of next words and probabilities.
# Print it for "the" and assert that it sums to 1.

def next_word_distribution(prev):
    distribution = {}

    for word in sorted(vocab):
        prob = bigram_prob(prev, word)

        if prob > 0:
            distribution[word] = prob

    return distribution


print("Next Word Distribution for 'the':")

distribution = next_word_distribution("the")

for word, prob in distribution.items():
    print(f"{word:<12} {prob:.4f}")

total_prob = sum(distribution.values())

assert abs(total_prob - 1.0) < 1e-9

print("\nSum of probabilities:", round(total_prob, 4))


# 3. Write bigram_sentence_prob(sentence) using
# the chain rule and print the result for both test sentences.

def bigram_sentence_prob(sentence):
    tokens = tokenize(sentence)

    probability = 1.0

    for i in range(1, len(tokens)):
        prev = tokens[i - 1]
        word = tokens[i]

        probability *= bigram_prob(prev, word)

    return probability


print("\nBigram Sentence Probabilities:")

for sentence in test:
    prob = bigram_sentence_prob(sentence)
    print(f"{sentence}: {prob:.4f}")


# 4. Write predict_next(prev) that returns the most
# probable next word (break ties alphabetically).
# Test it on at least five different words.

def predict_next(prev):
    distribution = next_word_distribution(prev)

    if not distribution:
        return None

    return sorted(
        distribution.items(),
        key=lambda x: (-x[1], x[0])
    )[0][0]


print("\nNext Word Predictions:")

words = ["the", "cat", "dog", "sat", "ate", "on"]

for word in words:
    prediction = predict_next(word)
    print(f"Previous word: {word:<8} Predicted next: {prediction}")

---
## Task 6: Trigram Model


In [ ]:
# 1. Write trigram_prob(w1, w2, w3) using
# P(w3 | w1, w2) = C(w1, w2, w3) / C(w1, w2).

def trigram_prob(w1, w2, w3):
    trigram_count = trigram_counts.get((w1, w2, w3), 0)
    bigram_count = bigram_counts.get((w1, w2), 0)

    if bigram_count == 0:
        return 0.0

    return trigram_count / bigram_count


# 2. Write trigram_sentence_prob(sentence).
# Use the bigram model for the first word given <s>
# and the trigram model for all other words.
# Print the result for both test sentences.

def trigram_sentence_prob(sentence):
    tokens = tokenize(sentence)

    probability = bigram_prob(tokens[0], tokens[1])

    for i in range(2, len(tokens)):
        w1 = tokens[i - 2]
        w2 = tokens[i - 1]
        w3 = tokens[i]

        probability *= trigram_prob(w1, w2, w3)

    return probability


print("Trigram Sentence Probabilities:")

for sentence in test:
    prob = trigram_sentence_prob(sentence)
    print(f"{sentence}: {prob:.4f}")


# 3. Print the number of distinct unigrams, bigrams
# and trigrams in a small table.

print("\nNumber of Distinct N-grams:")
print(f"{'N-gram Type':<15} {'Distinct Count':<15}")

print(f"{'Unigrams':<15} {len(unigram_counts):<15}")
print(f"{'Bigrams':<15} {len(bigram_counts):<15}")
print(f"{'Trigrams':<15} {len(trigram_counts):<15}")


# 4. For each test sentence, list its unseen bigrams
# and unseen trigrams, then print which model has
# more unseen N-grams.

print("\nUnseen N-grams in Test Sentences:")

for sentence in test:
    tokens = tokenize(sentence)

    bigrams = get_ngrams(tokens, 2)
    trigrams = get_ngrams(tokens, 3)

    unseen_bigram_list = [
        bg for bg in bigrams
        if bigram_counts.get(bg, 0) == 0
    ]

    unseen_trigram_list = [
        tg for tg in trigrams
        if trigram_counts.get(tg, 0) == 0
    ]

    print("\nSentence:", sentence)

    print("Unseen Bigrams:", unseen_bigram_list)
    print("Unseen Trigrams:", unseen_trigram_list)

    print("Total Unseen Bigrams:", len(unseen_bigram_list))
    print("Total Unseen Trigrams:", len(unseen_trigram_list))

    if len(unseen_bigram_list) > len(unseen_trigram_list):
        print("Bigram model has more unseen N-grams.")

    elif len(unseen_trigram_list) > len(unseen_bigram_list):
        print("Trigram model has more unseen N-grams.")

    else:
        print("Both models have equal unseen N-grams.")

---
## Task 7: Zero Probability Problem


In [ ]:
# 1. Write unseen_bigrams(sentence) that returns every
# bigram of the sentence that never occurs in training.

def unseen_bigrams(sentence):
    tokens = tokenize(sentence)
    bigrams = get_ngrams(tokens, 2)

    return [
        bg for bg in bigrams
        if bigram_counts.get(bg, 0) == 0
    ]


# 2. Run it on both test sentences and print the
# bigram MLE probability of each.
# Confirm that one of them is 0.

print("Unseen Bigrams and Bigram MLE Probabilities:")

for sentence in test:
    unseen = unseen_bigrams(sentence)
    prob = bigram_sentence_prob(sentence)

    print("\nSentence:", sentence)
    print("Unseen Bigrams:", unseen)
    print(f"Bigram MLE Probability: {prob:.4f}")

    if prob == 0:
        print("Confirmed: This sentence has zero probability.")


# 3. Count how many of the possible word pairs
# (words only, markers excluded) never occur in
# training, and print this as a percentage.

words = sorted(vocab - {"</s>", "<s>"})

total_pairs = len(words) * len(words)

unseen_pairs = 0

for w1 in words:
    for w2 in words:
        if bigram_counts.get((w1, w2), 0) == 0:
            unseen_pairs += 1

percentage = (unseen_pairs / total_pairs) * 100

print("\nPossible Word Pairs:", total_pairs)
print("Unseen Word Pairs:", unseen_pairs)
print(f"Percentage of Unseen Pairs: {percentage:.4f}%")


# 4. Write safe_log(p) that returns negative infinity
# for p = 0 instead of raising a math error.
# Use it to compute the log probability of the
# second test sentence.

import math

def safe_log(p):
    if p == 0:
        return float("-inf")
    return math.log(p)


sentence = test[1]

probability = bigram_sentence_prob(sentence)
log_probability = safe_log(probability)

print("\nSecond Test Sentence:", sentence)
print(f"Bigram Probability: {probability:.4f}")
print("Log Probability:", log_probability)

---
## Task 8: Smoothing


In [ ]:
# 1. Write bigram_prob_addone(prev, w) using
# (C(v, w) + 1) / (C(v) + V).

def bigram_prob_addone(prev, w):
    bigram_count = bigram_counts.get((prev, w), 0)

    # Count previous word, including <s>
    prev_count = sum(
        count for (w1, w2), count in bigram_counts.items()
        if w1 == prev
    )

    return (bigram_count + 1) / (prev_count + V)


# 2. Print P(sat | fish) and P(cat | the)
# under both MLE and add-one smoothing, side by side.

print("MLE vs Add-One Smoothing:")
print(f"{'Bigram':<20} {'MLE':<12} {'Add-One':<12}")

pairs = [("fish", "sat"), ("the", "cat")]

for prev, word in pairs:
    mle = bigram_prob(prev, word)
    addone = bigram_prob_addone(prev, word)

    print(f"{f'P({word}|{prev})':<20} {mle:<12.4f} {addone:<12.4f}")


# 3. Loop over every possible previous word and assert
# that its smoothed next-word probabilities over all V
# words sum to 1.

print("\nVerifying Smoothed Probability Sums:")

previous_words = sorted(vocab | {"<s>"})

for prev in previous_words:
    total = sum(
        bigram_prob_addone(prev, word)
        for word in vocab
    )

    assert abs(total - 1.0) < 1e-9

    print(f"{prev:<12} Sum = {total:.4f}")

print("\nAll smoothed distributions sum to 1!")


# 4. Write bigram_prob_addk(prev, w, k)
# and print P(sat | fish) for
# k = 1, 0.5, 0.1 and 0.01 in a table.

def bigram_prob_addk(prev, w, k):
    bigram_count = bigram_counts.get((prev, w), 0)

    prev_count = sum(
        count for (w1, w2), count in bigram_counts.items()
        if w1 == prev
    )

    return (bigram_count + k) / (prev_count + k * V)


print("\nAdd-k Smoothing for P(sat | fish):")
print(f"{'k':<10} {'Probability':<15}")

k_values = [1, 0.5, 0.1, 0.01]

for k in k_values:
    prob = bigram_prob_addk("fish", "sat", k)
    print(f"{k:<10} {prob:<15.4f}")

---
## Task 9: Perplexity and Model Comparison


In [ ]:
import math
from collections import Counter


# 1. Write perplexity(sentences, prob_fn) that returns
# infinity when any probability is 0.

def perplexity(sentences, prob_fn):
    total_log_prob = 0.0
    total_words = 0

    for sentence in sentences:
        tokens = tokenize(sentence)

        for i in range(1, len(tokens)):
            prev = tokens[i - 1]
            word = tokens[i]

            prob = prob_fn(prev, word)

            if prob <= 0:
                return float("inf")

            total_log_prob += math.log(prob)
            total_words += 1

    if total_words == 0:
        return float("inf")

    return math.exp(-total_log_prob / total_words)


# 2. Compute the perplexity of the two test sentences
# together for the bigram MLE model, the bigram add-one
# model and the unigram add-one model.
# Print one results table.

def unigram_prob_addone(w):
    return (unigram_counts.get((w,), 0) + 1) / (N + V)


mle_pp = perplexity(test, bigram_prob)

addone_bigram_pp = perplexity(test, bigram_prob_addone)

addone_unigram_pp = perplexity(
    test,
    lambda prev, w: unigram_prob_addone(w)
)

print("Perplexity Model Comparison:")
print(f"{'Model':<25} {'Perplexity':<15}")

print(f"{'Bigram MLE':<25} {mle_pp:.4f}")
print(f"{'Bigram Add-One':<25} {addone_bigram_pp:.4f}")
print(f"{'Unigram Add-One':<25} {addone_unigram_pp:.4f}")


# 3. Compute the perplexity of the add-k bigram model
# for k = 1, 0.5, 0.1 and 0.01.
# Print the table and the best k.

k_values = [1, 0.5, 0.1, 0.01]

k_results = {}

print("\nAdd-k Bigram Perplexity:")
print(f"{'k':<10} {'Perplexity':<15}")

for k in k_values:
    pp = perplexity(
        test,
        lambda prev, w: bigram_prob_addk(prev, w, k)
    )

    k_results[k] = pp
    print(f"{k:<10} {pp:<15.4f}")

best_k_test = min(k_results, key=k_results.get)

print("\nBest k on Test Set:", best_k_test)
print("Best Test Perplexity:",
      round(k_results[best_k_test], 4))


# 4. Train only on the first three training sentences
# (build V from all four) and pick the best k using the
# fourth sentence as validation data.
# Report the test perplexity with that k, and compare
# it with the best k found directly on the test set.

train_small = train[:3]
validation = [train[3]]

# Vocabulary stays built from all four training sentences
vocab_full = build_vocab(train)
V_full = len(vocab_full)

# Build bigram counts using only first three sentences
bigram_counts_small = Counter()

for sentence in train_small:
    tokens = tokenize(sentence)
    bigram_counts_small.update(get_ngrams(tokens, 2))


# Add-k probability using the smaller training corpus
def bigram_prob_addk_small(prev, w, k):
    bigram_count = bigram_counts_small.get((prev, w), 0)

    prev_count = sum(
        count for (w1, w2), count in bigram_counts_small.items()
        if w1 == prev
    )

    return (bigram_count + k) / (prev_count + k * V_full)


# Find the best k using validation sentence
validation_results = {}

print("\nValidation Perplexity:")
print(f"{'k':<10} {'Perplexity':<15}")

for k in k_values:
    pp = perplexity(
        validation,
        lambda prev, w: bigram_prob_addk_small(prev, w, k)
    )

    validation_results[k] = pp
    print(f"{k:<10} {pp:<15.4f}")

best_k_validation = min(
    validation_results,
    key=validation_results.get
)

print("\nBest k from Validation:", best_k_validation)


# Compute test perplexity using validation-selected k
# and the model trained on the first three sentences.

test_pp_validation = perplexity(
    test,
    lambda prev, w: bigram_prob_addk_small(
        prev, w, best_k_validation
    )
)

print("\nFinal Model Comparison:")
print(f"{'Model':<35} {'k':<10} {'Test Perplexity':<15}")

print(
    f"{'Validation-Selected Model':<35} "
    f"{best_k_validation:<10} "
    f"{test_pp_validation:<15.4f}"
)

print(
    f"{'Test-Selected Full-Training Model':<35} "
    f"{best_k_test:<10} "
    f"{k_results[best_k_test]:<15.4f}"
)